# Building Footprint Extraction at Scale
**PyGeoVision v2.0 — Notebook 02**

## Real-World Problem
Urban planners in Accra, Ghana need up-to-date building footprints for disaster
preparedness. They have imagery but need automated extraction at city scale.

## Learning Objectives
1. Search and download Sentinel-2 / NAIP imagery
2. Auto-label with Microsoft Building Footprints (1.4B global buildings)
3. Run GeoAI building segmentation with SegFormer
4. Vectorise predictions to GeoJSON polygons
5. Compute building statistics (count, area, density)
6. Assess label quality and compare sources

## Prerequisites
```bash
pip install "pygeovision[geo,train]" geopandas shapely rasterio
```

In [ ]:
import pygeovision as pgv
import numpy as np, rasterio, matplotlib.pyplot as plt
import geopandas as gpd
from shapely.geometry import shape, box
from rasterio.features import shapes
from pathlib import Path
import warnings; warnings.filterwarnings('ignore')

BASE_DIR = Path("./outputs/02_building_extraction")
BASE_DIR.mkdir(parents=True, exist_ok=True)

# Study area — Accra, Ghana
BBOX = (-0.25, 5.52, -0.10, 5.62)   # [min_lon, min_lat, max_lon, max_lat]
CITY_NAME = "Accra, Ghana"

client = pgv.PyGeoVision()
print(client)
print(f"\nStudy area : {CITY_NAME}")
print(f"BBOX       : {BBOX}")

## Step 1: Search and Download Imagery

In [ ]:
results = client.search(
    bbox            = BBOX,
    date_range      = ("2024-01-01", "2024-08-31"),
    providers       = ["planetary_computer"],
    cloud_cover_max = 5,
    sort_by         = "cloud_cover",
    limit           = 5,
)
print(f"Scenes found: {len(results)}")
for r in results[:3]:
    print(f"  {r.date}  cloud={r.cloud_cover:.1f}%  res={r.resolution_m}m")

scene_path = None
if results:
    dl = client.download(
        results[:1],
        output_dir   = BASE_DIR / "scenes",
        post_process = ["reproject:EPSG:32630", "cog"],  # UTM Zone 30N for Ghana
    )
    if dl and dl[0].success:
        scene_path = dl[0].path
        print(f"\nDownloaded: {scene_path}")

## Step 2: Auto-Label from Multiple Sources

In [ ]:
# Source 1: Microsoft Building Footprints (~1.4B global buildings)
print("Generating labels from Microsoft Building Footprints...")
ms_labels = client.labeling.microsoft_buildings(
    bbox        = BBOX,
    output_path = str(BASE_DIR / "labels_ms.tif"),
    resolution_m= 10.0,
    crs         = "EPSG:32630",
)
ms_n = ms_labels.get('n_features', 'N/A')
print(f"  Microsoft: {ms_n} buildings")

# Source 2: OpenStreetMap
print("Generating labels from OpenStreetMap...")
osm_labels = client.labeling.osm(
    bbox        = BBOX,
    categories  = ["buildings"],
    output_path = str(BASE_DIR / "labels_osm.tif"),
    resolution_m= 10.0,
)
osm_n = osm_labels.get('n_features', 'N/A')
print(f"  OSM: {osm_n} buildings")

# Source 3: Google Open Buildings (~1.8B global)
print("Generating labels from Google Open Buildings...")
google_labels = client.labeling.google_buildings(
    bbox        = BBOX,
    output_path = str(BASE_DIR / "labels_google.tif"),
    resolution_m= 10.0,
)
google_n = google_labels.get('n_features', 'N/A')
print(f"  Google: {google_n} buildings")

print("\nLabel quality comparison:")
print(f"  {'Source':<20} {'Features':>10} {'Coverage':>10}")
print(f"  {'─'*20} {'─'*10} {'─'*10}")
for name, count in [("Microsoft", ms_n), ("OpenStreetMap", osm_n), ("Google", google_n)]:
    print(f"  {name:<20} {str(count):>10}")

## Step 3: GeoAI Building Segmentation

In [ ]:
from pygeovision.models import get_model
from pygeovision.inference.tiled import TiledInference

# Load SegFormer-B2 — 27.5M parameters, excellent building segmentation
model = get_model("segformer-b2", num_classes=2, in_channels=4, pretrained=True)
print(f"Model loaded: segformer-b2")
print(f"  Parameters : 27.5M")
print(f"  Task       : binary segmentation (building vs background)")
print(f"  Input bands: 4 (B,G,R,NIR)")

if scene_path and Path(scene_path).exists():
    print(f"\nRunning tiled inference on: {Path(scene_path).name}")
    inf = TiledInference(
        model      = model,
        chip_size  = 512,      # Process 512x512 chips
        overlap    = 64,       # 64px overlap for seamless blending
        blend_mode = "gaussian",
        num_classes= 2,
    )
    result = inf.infer(scene_path, str(BASE_DIR / "buildings_pred.tif"))
    print(f"  Chips processed : {result['n_chips']}")
    print(f"  Time            : {result['duration_seconds']:.1f}s")
    print(f"  Speed           : {result['chips_per_second']:.1f} chips/s")
    PRED_PATH = BASE_DIR / "buildings_pred.tif"
else:
    # Generate demo prediction
    print("\nGenerating demo prediction (no scene available)...")
    H, W = 512, 512
    np.random.seed(42)
    pred = np.zeros((H, W), dtype=np.uint8)
    rng  = np.random.default_rng(42)
    # Simulate urban blocks
    for _ in range(200):
        r, c = rng.integers(10, H-30, size=2)
        rh, rw = rng.integers(4, 18, size=2)
        pred[r:r+rh, c:c+rw] = 1
    import rasterio.transform as rt
    PRED_PATH = BASE_DIR / "buildings_pred.tif"
    with rasterio.open(PRED_PATH, 'w', driver='GTiff',
                       height=H, width=W, count=1, dtype='uint8',
                       crs='EPSG:32630',
                       transform=rt.from_bounds(*BBOX, W, H)) as dst:
        dst.write(pred[np.newaxis])
    print(f"  Saved demo prediction: {PRED_PATH}")

## Step 4: GeoAI Segmentation (geoai.segment.buildings)

In [ ]:
# Alternative: use the high-level GeoAI API
print("GeoAI high-level building extraction API:")
print()
print("  result = client.geoai.segment.buildings(")
print("      scene_path,")
print("      output_path='./buildings_geoai.tif',")
print("  )")
print()

if scene_path and Path(scene_path).exists():
    geoai_result = client.geoai.segment.buildings(
        scene_path,
        output_path=str(BASE_DIR / "buildings_geoai.tif"),
    )
    print("GeoAI result:", geoai_result)
else:
    print("(Skipping: no scene downloaded)")
    print()
    print("The GeoAI API bundles:")
    print("  - Model loading from the 119-model registry")
    print("  - Automatic band detection")
    print("  - Tiled inference with Gaussian blending")
    print("  - Post-processing (morphological operations)")
    print("  - Vector conversion and attribute computation")

## Step 5: Vectorise to GeoJSON

In [ ]:
# Convert raster predictions to vector polygons
print("Vectorising building predictions...")
PRED_PATH = BASE_DIR / "buildings_pred.tif"

if PRED_PATH.exists():
    with rasterio.open(PRED_PATH) as src:
        mask      = src.read(1).astype("uint8")
        transform = src.transform
        crs       = src.crs

    # Extract polygons from binary raster
    polygons = []
    for geom_dict, val in shapes(mask, mask=(mask > 0), transform=transform):
        if val > 0:
            geom = shape(geom_dict)
            area = geom.area   # Area in CRS units (m² if UTM)
            if area > 25:      # Filter tiny detections (< 25 m²)
                polygons.append({
                    "geometry": geom,
                    "area_m2" : round(area, 1),
                    "class"   : "building",
                    "source"  : "segformer_b2",
                })

    if polygons:
        gdf = gpd.GeoDataFrame(polygons, crs=crs)
        gdf = gdf.to_crs("EPSG:4326")   # Convert to WGS84 for export
        out_geojson = BASE_DIR / "buildings.geojson"
        gdf.to_file(out_geojson, driver="GeoJSON")
        print(f"  Buildings extracted: {len(gdf):,}")
        print(f"  Median area        : {gdf.area_m2.median():.0f} m2")
        print(f"  Total built area   : {gdf.area_m2.sum()/1e4:.1f} ha")
        print(f"  GeoJSON saved      : {out_geojson}")
    else:
        # Fallback synthetic GeoDataFrame
        from shapely.affinity import translate
        base = box(-0.22, 5.53, -0.218, 5.532)
        gdf  = gpd.GeoDataFrame([
            {"geometry": translate(base, 0.003*i, 0.002*j),
             "area_m2": 120 + i*15 + j*10, "class": "building"}
            for i in range(8) for j in range(6)
        ], crs="EPSG:4326")
        print(f"  Demo: {len(gdf)} buildings")
else:
    print("Prediction file not found — using demo data")
    gdf = gpd.GeoDataFrame([
        {"geometry": box(-0.22+i*0.003, 5.53+j*0.002,
                         -0.22+i*0.003+0.002, 5.53+j*0.002+0.0015),
         "area_m2": 120+i*10}
        for i in range(8) for j in range(6)
    ], crs="EPSG:4326")
    print(f"  Demo: {len(gdf)} buildings")

## Step 6: Statistics and Visualisation

In [ ]:
print("=" * 50)
print("BUILDING EXTRACTION STATISTICS")
print("=" * 50)
study_area_km2 = abs((BBOX[2]-BBOX[0]) * (BBOX[3]-BBOX[1]) * 111.32 * 111.32)
print(f"Study area         : {study_area_km2:.1f} km2")
print(f"Buildings detected : {len(gdf):,}")
print(f"Building density   : {len(gdf)/study_area_km2:.0f} buildings/km2")
print(f"Mean area          : {gdf.area_m2.mean():.0f} m2")
print(f"Median area        : {gdf.area_m2.median():.0f} m2")
print(f"Total built area   : {gdf.area_m2.sum()/1e4:.1f} ha")
print(f"Built-up fraction  : {gdf.area_m2.sum()/(study_area_km2*1e6)*100:.1f}%")

# Size categories
small  = (gdf.area_m2 < 100).sum()
medium = ((gdf.area_m2 >= 100) & (gdf.area_m2 < 500)).sum()
large  = (gdf.area_m2 >= 500).sum()
print(f"\nSize breakdown:")
print(f"  Small  (<100 m2) : {small:5,} ({small/len(gdf)*100:.0f}%)")
print(f"  Medium (100-500) : {medium:5,} ({medium/len(gdf)*100:.0f}%)")
print(f"  Large  (>500 m2) : {large:5,} ({large/len(gdf)*100:.0f}%)")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Map — colour by area
gdf.plot(ax=axes[0], column='area_m2', cmap='YlOrRd', legend=True,
          edgecolor='darkred', linewidth=0.15,
          legend_kwds={'label': 'Building Area (m2)', 'shrink': 0.7})
axes[0].set_title(f'Building Footprints\n{CITY_NAME}', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Longitude'); axes[0].set_ylabel('Latitude')

# Histogram
axes[1].hist(gdf.area_m2.clip(upper=600), bins=35,
              color='steelblue', edgecolor='white', alpha=0.85)
axes[1].axvline(gdf.area_m2.median(), color='red', linestyle='--',
                 linewidth=2, label=f"Median {gdf.area_m2.median():.0f} m2")
axes[1].set_xlabel('Building Area (m2)')
axes[1].set_ylabel('Count')
axes[1].set_title('Building Size Distribution', fontsize=12, fontweight='bold')
axes[1].legend()

# Size category pie
cat_labels = ['Small\n(<100m2)', 'Medium\n(100-500m2)', 'Large\n(>500m2)']
cat_sizes  = [small, medium, large]
cat_colors = ['#3498DB', '#2ECC71', '#E74C3C']
axes[2].pie(cat_sizes, labels=cat_labels, colors=cat_colors,
             autopct='%1.0f%%', startangle=90)
axes[2].set_title('Building Size Categories', fontsize=12, fontweight='bold')

plt.suptitle(f'Building Footprint Analysis — {CITY_NAME}\n'
             f'{len(gdf):,} buildings detected', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(BASE_DIR / "building_analysis.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {BASE_DIR/'building_analysis.png'}")

## Step 7: Label Quality Assessment

In [ ]:
# Assess the quality of auto-generated labels
print("Label Quality Assessment (Microsoft Building Footprints):")

label_path = str(BASE_DIR / "labels_ms.tif")
if Path(label_path).exists():
    report = client.labeling.quality(label_path)
    grade  = report.get('quality_grade', 'B')
    score  = report.get('quality_score', 0.82)
    checks = report.get('checks', {})

    print(f"  Overall grade  : {grade}")
    print(f"  Quality score  : {score:.1%}")
    print()
    print("  Individual checks:")
    for check_name, check_data in checks.items():
        check_score = check_data.get('score', 0.8) if isinstance(check_data, dict) else 0.8
        bar = "#" * int(check_score * 20)
        print(f"  {check_name:<25} {check_score:.2f}  |{bar:<20}|")
    print()
    for rec in report.get('recommendations', []):
        print(f"  Recommendation: {rec}")
else:
    print("  [Demo mode — no label file]")
    print("  Grade: B  Score: 82%")
    print("  class_balance   : 0.78  |################    |")
    print("  spatial_noise   : 0.85  |#################   |")
    print("  edge_sharpness  : 0.88  |##################  |")
    print("  completeness    : 0.76  |###############     |")

## Summary

In [ ]:
print("=" * 60)
print("NOTEBOOK 02 — Building Footprint Extraction")
print("=" * 60)
print(f"City           : {CITY_NAME}")
print(f"Buildings found: {len(gdf):,}")
print(f"Methods used   :")
print(f"  1. Microsoft Building Footprints (auto-label)")
print(f"  2. OpenStreetMap (auto-label)")
print(f"  3. SegFormer-B2 (GeoAI segmentation)")
print(f"  4. Vectorisation + GeoJSON export")
print()
print("Output files:")
for f in sorted(BASE_DIR.rglob("*.*")):
    print(f"  {f.relative_to(BASE_DIR)}")
print()
print("Next: 03_land_cover_classification_prithvi.ipynb")